# Stage C — read-only checkpoint reload/resume verification
This notebook never writes to the successful c9d run. It loads the same checkpoint twice, continues each restored state by one optimizer step, compares deterministic scientific-state hashes, and writes a separate verification artifact.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='dd566c1af9120fd5f4160e48a29746b6231acbaa'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
SOURCE_RUN='c9d_25m_adaptive_stability_500steps_preconditioned'
OUTPUT_RUN='c9d_checkpoint_resume_verification'
EXPECTED_STEP=500
GRADIENT_CLIP_NORM=0.5

In [ ]:
from pathlib import Path
from google.colab import drive
import json, subprocess, sys
mountpoint=Path('/content/drive')
try:
    if not (mountpoint/'MyDrive').is_dir(): drive.mount(str(mountpoint),force_remount=True,timeout_ms=120000)
except ValueError as error:
    raise RuntimeError('Google Drive did not mount. Restart the runtime, reconnect Drive, authorize it, and rerun this cell.') from error
if not (mountpoint/'MyDrive').is_dir(): raise RuntimeError('Google Drive is not ready at /content/drive/MyDrive.')
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
import torch
device_name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no CUDA device'
if 'A100' not in device_name.upper(): raise RuntimeError(f'Notebook 03b requires the same A100 execution class; Colab assigned {device_name}.')
selection_path=Path(DRIVE_ROOT)/'runs'/'c1_tokenizers_cpu'/'tokenizer_selection.json'
selection=json.loads(selection_path.read_text(encoding='utf-8'))
selected=selection['selected_tokenizer']
dataset=Path(DRIVE_ROOT)/'stage_c_dataset'/'ordered_streams'/selected
if not (dataset/'token_stream_manifest.json').is_file(): raise FileNotFoundError(f'Missing dataset manifest: {dataset}')
checkpoint=Path(DRIVE_ROOT)/'runs'/SOURCE_RUN/'adaptive'/'latest.pt'
if not checkpoint.is_file(): raise FileNotFoundError(f'Missing source checkpoint: {checkpoint}')
output=Path(DRIVE_ROOT)/'runs'/OUTPUT_RUN
output.mkdir(parents=True,exist_ok=True)
report=output/'resume_verification.json'
PROTOCOL=repo/'studies'/'stage_c_ecoli_escherichia_medium_25m_v1'/'protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study'/'stage_c_ecoli_escherichia_medium_25m_v1'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
print('Read-only source checkpoint:',checkpoint)
print('Separate verification output:',output)

In [ ]:
command=['seqtrainer-titans-stage-c-resume-verify','--dataset-dir',str(dataset),'--checkpoint',str(checkpoint),'--output',str(report),'--device','cuda','--gradient-clip-norm',str(GRADIENT_CLIP_NORM),'--expected-step',str(EXPECTED_STEP),'--expected-code-commit','762240f40cebb4e5d7202714c98613b879739694']
subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(output),'--label','resume_verify','--repo',str(repo),'--',*command],check=True)
result=json.loads(report.read_text(encoding='utf-8'))
if result['status'] != 'passed': raise RuntimeError(json.dumps(result,indent=2))
subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id','calibration_50k','--evidence-tier','engineering','--artifact',str(output)],check=True)
print(json.dumps({
    'status':result['status'],
    'source_checkpoint_unchanged':result['read_only_source_checkpoint'],
    'deterministic_continuation':result['deterministic_continuation'],
    'loaded_step':result['first_continuation']['loaded_optimizer_step'],
    'continued_step':result['first_continuation']['continued_optimizer_step'],
    'checkpoint_sha256':result['checkpoint_sha256_before'],
    'code_commit_warning':result['code_commit_warning'],
},indent=2))
print('SHARE THIS DIRECTORY:',output)